In [346]:
# fetch data and explore using the instructions here: https://archive.ics.uci.edu/dataset/336/chronic%2Bkidney%2Bdisease
! pip install ucimlrepo
from ucimlrepo import fetch_ucirepo 

chronic_kidney_disease = fetch_ucirepo(id=336) 

print('\nvariables:')
display(chronic_kidney_disease.variables)


variables:


,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,NaN,year,yes
1,bp,Feature,Integer,NaN,blood pressure,mm/Hg,yes
2,sg,Feature,Categorical,NaN,specific gravity,NaN,yes
3,al,Feature,Categorical,NaN,albumin,NaN,yes
4,su,Feature,Categorical,NaN,sugar,NaN,yes
5,rbc,Feature,Binary,NaN,red blood cells,NaN,yes
6,pc,Feature,Binary,NaN,pus cell,NaN,yes
7,pcc,Feature,Binary,NaN,pus cell clumps,NaN,yes
8,ba,Feature,Binary,NaN,bacteria,NaN,yes
9,bgr,Feature,Integer,NaN,blood glucose random,mgs/dl,yes


In [347]:
# explore features 
print('features:')
display(chronic_kidney_disease.data.features)

# explore target 
print('targets:') 
display(chronic_kidney_disease.data.targets)

features:


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,36.0,1.2,NaN,NaN,15.4,44.0,7800.0,5.2,yes,yes,no,good,no,no
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,18.0,0.8,NaN,NaN,11.3,38.0,6000.0,NaN,no,no,no,good,no,no
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,NaN,NaN,9.6,31.0,7500.0,NaN,no,yes,no,poor,no,yes
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9,yes,no,no,poor,yes,yes
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,NaN,NaN,11.6,35.0,7300.0,4.6,no,no,no,good,no,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,55.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,140.0,49.0,0.5,150.0,4.9,15.7,47.0,6700.0,4.9,no,no,no,good,no,no
396,42.0,70.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,75.0,31.0,1.2,141.0,3.5,16.5,54.0,7800.0,6.2,no,no,no,good,no,no
397,12.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,100.0,26.0,0.6,137.0,4.4,15.8,49.0,6600.0,5.4,no,no,no,good,no,no
398,17.0,60.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,114.0,50.0,1.0,135.0,4.9,14.2,51.0,7200.0,5.9,no,no,no,good,no,no


targets:


,class
0,ckd
1,ckd
2,ckd
3,ckd
4,ckd
...,...
395,notckd
396,notckd
397,notckd
398,notckd


In [348]:
import pandas as pd

X = chronic_kidney_disease.data.features
y = chronic_kidney_disease.data.targets

pd.set_option('display.max_columns', None)

print('X before mapping') 
display(X.head(5))
print('y before mapping') 
display(y)

# map string values to numeric values - needed for sklearn compatibility 
X['rbc'] = X['rbc'].map({'normal': 1, 'abnormal': 0})
X['pc'] = X['pc'].map({'normal': 1, 'abnormal': 0})
X['pcc'] = X['pcc'].map({'present': 1, 'notpresent': 0})
X['ba'] = X['ba'].map({'present': 1, 'notpresent': 0})
X['htn'] = X['htn'].map({'yes': 1, 'no': 0})
X['dm'] = X['dm'].map({'yes': 1, 'no': 0})
X['cad'] = X['cad'].map({'yes': 1, 'no': 0})
X['appet'] = X['appet'].map({'good': 1, 'poor': 0})
X['pe'] = X['pe'].map({'yes': 1, 'no': 0})
X['ane'] = X['ane'].map({'yes': 1, 'no': 0})
y['class'] = y['class'].map({'ckd': 1, 'notckd': 0})

print('X after mapping') 
display(X.head(5))
print('y after mapping') 
display(y)

# find rows with NaN values in the target variable — sklearn needs each row to have a defined target value 
non_nan_rows = y['class'].notna()

# filter out rows with NaN values in the target variable 
X = X[non_nan_rows]

# for the features, fill NaN values with the mode of each column. While typically mode isn't chosen for non-categorical 
# numeric values, we have many categorical features in this dataset, so mode is a reasonable choice. Ideally, we use mode 
# for categorical features and median or mean for non-categorical numeric features. Empirically, this did not cause issues. 
# Source: https://www.linkedin.com/pulse/when-use-mean-median-mode-handling-missing-values-data-ahmed-tebje/ 
X = X.fillna(X.mode().iloc[0])

# retain only non-NaN rows in the target variable and flatten from (400,1) to (400,) for sklearn compatibility
y = y.loc[non_nan_rows, 'class']

X before mapping


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,36.0,1.2,NaN,NaN,15.4,44.0,7800.0,5.2,yes,yes,no,good,no,no
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,18.0,0.8,NaN,NaN,11.3,38.0,6000.0,NaN,no,no,no,good,no,no
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,NaN,NaN,9.6,31.0,7500.0,NaN,no,yes,no,poor,no,yes
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9,yes,no,no,poor,yes,yes
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,NaN,NaN,11.6,35.0,7300.0,4.6,no,no,no,good,no,no


y before mapping


,class
0,ckd
1,ckd
2,ckd
3,ckd
4,ckd
...,...
395,notckd
396,notckd
397,notckd
398,notckd


X after mapping


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane
0,48.0,80.0,1.020,1.0,0.0,NaN,1.0,0.0,0.0,121.0,36.0,1.2,NaN,NaN,15.4,44.0,7800.0,5.2,1.0,1.0,0.0,1.0,0.0,0.0
1,7.0,50.0,1.020,4.0,0.0,NaN,1.0,0.0,0.0,NaN,18.0,0.8,NaN,NaN,11.3,38.0,6000.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0
2,62.0,80.0,1.010,2.0,3.0,1.0,1.0,0.0,0.0,423.0,53.0,1.8,NaN,NaN,9.6,31.0,7500.0,NaN,0.0,1.0,0.0,0.0,0.0,1.0
3,48.0,70.0,1.005,4.0,0.0,1.0,0.0,1.0,0.0,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9,1.0,0.0,0.0,0.0,1.0,1.0
4,51.0,80.0,1.010,2.0,0.0,1.0,1.0,0.0,0.0,106.0,26.0,1.4,NaN,NaN,11.6,35.0,7300.0,4.6,0.0,0.0,0.0,1.0,0.0,0.0


y after mapping


,class
0,1.0
1,1.0
2,1.0
3,1.0
4,1.0
...,...
395,0.0
396,0.0
397,0.0
398,0.0


In [349]:
from sklearn.model_selection import train_test_split

# split the dataset into training and testing sets. 20% for testing, 80% for training
# random state is for reproducibility of results 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)

X_train shape: (318, 24)
y_train shape: (318,)
X_test shape: (80, 24)
y_test shape: (80,)


In [350]:
# import the classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [351]:
# train and evaluate models

# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
logistic_regression_model = LogisticRegression(max_iter=1000)
logistic_regression_model.fit(X_train, y_train)
log_regression_preds = logistic_regression_model.predict(X_test)

# https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
tree_model = DecisionTreeClassifier(random_state=10)
tree_model.fit(X_train, y_train)
tree_preds = tree_model.predict(X_test)

# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
forest_model = RandomForestClassifier(random_state=10)
forest_model.fit(X_train, y_train)
forest_preds = forest_model.predict(X_test)

# https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html
mlp_model = MLPClassifier(hidden_layer_sizes=(32, 8), max_iter=1000, random_state=10)
mlp_model.fit(X_train, y_train)
mlp_preds = mlp_model.predict(X_test)

# evaluate models
print('Logistic Regression:', accuracy_score(y_test, log_regression_preds))
print('Decision Tree:', accuracy_score(y_test, tree_preds))
print('Random Forest:', accuracy_score(y_test, forest_preds))
print('MLP:', accuracy_score(y_test, mlp_preds))

Logistic Regression: 0.95
Decision Tree: 0.9875
Random Forest: 0.9875
MLP: 0.875


/opt/anaconda3/envs/healthcare/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [352]:
# try a few random_states to see if the results are consistent 
for seed in [11, 12, 13, 14]:
    # split the dataset into training and testing sets. 20% for testing, 80% for training
    # random state is for reproducibility of results 
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)


    # train and evaluate models

    # https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
    logistic_regression_model = LogisticRegression(max_iter=1000)
    logistic_regression_model.fit(X_train, y_train)
    log_regression_preds = logistic_regression_model.predict(X_test)

    # https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
    tree_model = DecisionTreeClassifier(random_state=seed)
    tree_model.fit(X_train, y_train)
    tree_preds = tree_model.predict(X_test)

    # https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
    forest_model = RandomForestClassifier(random_state=seed)
    forest_model.fit(X_train, y_train)
    forest_preds = forest_model.predict(X_test)

    # https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html
    mlp_model = MLPClassifier(hidden_layer_sizes=(32, 8), max_iter=1000, random_state=seed)
    mlp_model.fit(X_train, y_train)
    mlp_preds = mlp_model.predict(X_test)

    print('seed:', seed), '\n'
    print('Logistic Regression:', accuracy_score(y_test, log_regression_preds))
    print('Decision Tree:', accuracy_score(y_test, tree_preds))
    print('Random Forest:', accuracy_score(y_test, forest_preds))
    print('MLP:', accuracy_score(y_test, mlp_preds))

/opt/anaconda3/envs/healthcare/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


seed: 11
Logistic Regression: 0.95
Decision Tree: 0.95
Random Forest: 1.0
MLP: 0.725


/opt/anaconda3/envs/healthcare/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


seed: 12
Logistic Regression: 0.9625
Decision Tree: 0.9875
Random Forest: 1.0
MLP: 0.8625


/opt/anaconda3/envs/healthcare/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


seed: 13
Logistic Regression: 0.9375
Decision Tree: 0.95
Random Forest: 1.0
MLP: 0.7875
seed: 14
Logistic Regression: 1.0
Decision Tree: 0.95
Random Forest: 1.0
MLP: 0.925


/opt/anaconda3/envs/healthcare/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [353]:
# feature_importances_ extracts how much each feature contributed to predictions 
# https://inria.github.io/scikit-learn-mooc/python_scripts/dev_features_importance.html 
feature_importance = pd.DataFrame({'feature': X.columns,'importance': forest_model.feature_importances_}).sort_values('importance', ascending=False)
print(feature_importance.head(5))

   feature  importance
11      sc    0.158109
15     pcv    0.152803
14    hemo    0.140276
2       sg    0.129910
3       al    0.084479
